# Regresión Lineal - Precios de Propiedades en CABA

Implementación de regresión lineal desde cero y con scikit-learn para 
predecir el precio de propiedades en CABA.

## Features y target

**Features (X)**: `metros`, `ambientes`, `banos`, `expensas`    
**Target (y)**: `precio` en USD

## Estructura                                                     
1. Setup e imports  
2. Carga                                               
3. Preparación de los datos (carga, split train/val/test)         
4. Feature scaling (z-score normalization)                        
5. Regresión lineal desde scratch (función de costo + gradient descent)
6. Regularización desde scratch (Ridge / L2),                      
7. Evaluación (MSE, RMSE, MAE, R²)                                
8. Implementación con scikit-learn

### 1. Setup/Imports

In [5]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

### 2. Carga

In [6]:
df = pd.read_csv("../data/processed/zonaprop_clean.csv")
print(f"Filas: {len(df):,}")
df[["precio", "metros", "ambientes", "banos", "expensas"]].head()

Filas: 32,585


,precio,metros,ambientes,banos,expensas
0,530000.0,172.0,4.0,3.0,770000
1,170000.0,73.0,4.0,1.0,0
2,120000.0,54.0,3.0,1.0,260000
3,220000.0,68.0,3.0,2.0,380073
4,84100.0,55.0,1.0,1.0,110000


### 3. Split train/val/test

In [7]:
features = ["metros", "ambientes", "banos", "expensas"]

X = df[features].values
y = df["precio"].values

np.random.seed(42)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
n = len(X)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
X_train, y_train = X[:n_train],           y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],     y[n_train+n_val:]
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

Train: (22809, 4)
Val:   (4887, 4)
Test:  (4889, 4)


### 4. Feature scaling (Z-score normalization)
  
Transformamos cada feature para que tenga media 0 y desvío estándar 1.
Los parámetros μ y σ se calculan solo sobre train y se aplican a val y test para no filtrar información futura al modelo.

  $$x' = \frac{x - \mu}{\sigma}$$


In [8]:
mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

X_train_s = (X_train - mu) / sigma
X_val_s   = (X_val   - mu) / sigma
X_test_s  = (X_test  - mu) / sigma

print("Medias por feature (train escalado):")
print(X_train_s.mean(axis=0).round(6))
print("\nDesvíos por feature (train escalado):")
print(X_train_s.std(axis=0).round(6))

Medias por feature (train escalado):
[ 0.  0. -0.  0.]

Desvíos por feature (train escalado):
[1. 1. 1. 1.]


  ### 5. Regresión lineal desde scratch

  El modelo lineal predice el precio como una combinación de las features:
  
  $$\hat{y} = w_1x_1 + w_2x_2 + ... + w_nx_n + b = \mathbf{w}\mathbf{X} + b$$
  
  donde $\mathbf{w}$ son los pesos de cada feature y $b$ el bias. El objetivo es encontrar
  los valores de $\mathbf{w}$ y $b$ que minimicen el error entre las predicciones y los
  valores reales.
  
  Para eso se usan tres funciones encadenadas:
  
  **`compute_cost`** — mide qué tan mal está el modelo en un momento dado usando el error
  cuadrático medio:
  
  $$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$
  
  **`compute_gradient`** — calcula en qué dirección mover $\mathbf{w}$ y $b$ para reducir
  $J$. Son las derivadas parciales de la función de costo:
  
  $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{y} - y) \qquad
  \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$
  
  **`gradient_descent`** — el loop principal. En cada iteración usa el gradiente para
  actualizar los parámetros moviéndolos en la dirección contraria al error:
  
  $$w := w - \alpha \frac{\partial J}{\partial w} \qquad b := b - \alpha \frac{\partial
  J}{\partial b}$$
  
  $\alpha$ es el learning rate y controla el tamaño de cada paso. Al final de todas las
  iteraciones devuelve $\mathbf{w}$ y $b$ óptimos.
  


#### 5.1 `compute_cost`

In [10]:
def compute_cost(X, y, w, b): 
    m = X.shape[0]
    y_hat = X @ w +b 
    cost = (1 / (2 * m)) * np.sum((y_hat - y) ** 2) 
    return cost

#### 5.2 `compute_gradient`

In [11]:
def compute_gradient(X, y, w, b):
     m = len(y)
     y_hat = X @ w + b
     error = y_hat - y
 
     dw = (1 / m) * X.T @ error
     db = (1 / m) * np.sum(error)
     return dw, db